In [16]:
import chromadb

import numpy as np
import pandas as pd

from tqdm import tqdm

In [2]:
from embedder.ChromaConnector import (ChromaConnection, 
                                      VectorDBConnectionConfig, 
                                      VectorDBInstance)

In [3]:
data_path = "/home/jovyan/work/RAG-project-SMILES-2024-/data/squadv2/chroma/dbs/v3/"

In [4]:
db_contexts_path = "/home/jovyan/work/RAG-project-SMILES-2024-/data/squadv2/chroma/dbs/v3"
db_contexts_info = {'db': 'squadv2', 'table': 'contexts'}
vdb_contexts_conf = VectorDBConnectionConfig(path=db_contexts_path, db_info=db_contexts_info)
connector_contexts = ChromaConnection(config=vdb_contexts_conf)

In [5]:
connector_contexts.count_items()

19029

In [6]:
db_query_path = "/home/jovyan/work/RAG-project-SMILES-2024-/data/squadv2/chroma/dbs/v3"
db_query_info = {'db': 'squadv2', 'table': 'questions'}
vdb_query_conf = VectorDBConnectionConfig(path=db_query_path, db_info=db_query_info)
connector_query = ChromaConnection(config=vdb_query_conf)

In [7]:
connector_query.count_items()

86821

In [15]:
num_queries = connector_query.count_items()

In [20]:
batch_size = 5000
n_results = 15

In [21]:
for i in tqdm(range(0, num_queries, batch_size)):
    ids = [f'id{j}' for j in range(i, min(i + batch_size, num_queries))]
    queries = connector_query.read(ids=ids)
    scores = connector_contexts.retrieve(query_instances=queries, n_results=n_results, includes=[])
    df = pd.DataFrame(columns=["query_id", "contexts_ids", "cos_dists"])
    df["query_id"] = ids
    df["contexts_ids"] = [[ctx[1].id for ctx in ctxs] for ctxs in scores]
    df["cos_dists"] = [[ctx[0] for ctx in ctxs] for ctxs in scores]
    df.to_csv(f'cosine_scores_batch{i}.csv', index=False)

100%|██████████| 18/18 [01:18<00:00,  4.35s/it]


In [22]:
dfs = []
for df_name in [f'cosine_scores_batch{i}.csv' for i in range(0, num_queries, batch_size)]:
    dfs.append(pd.read_csv(df_name))
dfs = pd.concat(dfs)

In [23]:
dfs.shape

(86821, 3)

In [24]:
dfs.head()

,query_id,contexts_ids,cos_dists
0,id0,"['id10', 'id0', 'id45', 'id53', 'id2', 'id26',...","[0.15416234731674194, 0.15581238269805908, 0.1..."
1,id1,"['id4', 'id0', 'id5', 'id3', 'id64', 'id50', '...","[0.15261220932006836, 0.16854727268218994, 0.1..."
2,id2,"['id10', 'id0', 'id1', 'id53', 'id12', 'id23',...","[0.13007789850234985, 0.13846325874328613, 0.1..."
3,id3,"['id8', 'id6114', 'id19', 'id6115', 'id6', 'id...","[0.16860461235046387, 0.17995333671569824, 0.1..."
4,id4,"['id84', 'id67', 'id13616', 'id94', 'id66', 'i...","[0.16440743207931519, 0.17258894443511963, 0.1..."


In [25]:
dfs.tail()

,query_id,contexts_ids,cos_dists
1816,id86816,"['id18973', 'id18917', 'id18932', 'id18934', '...","[0.11931037902832031, 0.1748185157775879, 0.18..."
1817,id86817,"['id3367', 'id3362', 'id3360', 'id3358', 'id16...","[0.18268996477127075, 0.18880540132522583, 0.1..."
1818,id86818,"['id18973', 'id18916', 'id18927', 'id18923', '...","[0.13600891828536987, 0.14367049932479858, 0.1..."
1819,id86819,"['id18973', 'id18917', 'id18934', 'id18924', '...","[0.1315150260925293, 0.16991901397705078, 0.17..."
1820,id86820,"['id225', 'id724', 'id14146', 'id15676', 'id11...","[0.20946866273880005, 0.21423524618148804, 0.2..."


In [26]:
dfs.to_csv("cosine_scores_squad2.csv", index=False)